## 基本的なRAGシステムの構築
- 研修用のコードでは、以下のエラーとなったため、代替えコードで試す。

``` bash
--> 549     raise ValueError(
    550         "ChatMode.REACT and ChatMode.OPENAI are now deprecated and removed. "
    551         "Please use the ReActAgent or FunctionAgent classes from llama_index.core.agent.workflow "
    552         "to create an agent with a query engine tool."
    553     )
```

- 自環境のバージョンは以下

``` bash
llama-index 0.14.15
llama-index-core 0.14.15
llama-index-llms-openai 0.6.19
```


In [2]:
import sys
!{sys.executable} -m pip install llama-index


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai import OpenAI

# 環境変数の取得
load_dotenv("../.env")
os.environ['OPENAI_API_KEY']  = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"


In [7]:
# ドキュメントからテキスト情報を読込
documents = SimpleDirectoryReader('./data/text').load_data()

# インデックスの構築
index = VectorStoreIndex.from_documents(documents)


2026-02-24 12:50:04,207 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [8]:
# Chat Engineの作成
llm = OpenAI(model=MODEL_NAME)
chat_engine = index.as_chat_engine(chat_mode="openai", llm=llm, verbose=True)


ValueError: ChatMode.REACT and ChatMode.OPENAI are now deprecated and removed. Please use the ReActAgent or FunctionAgent classes from llama_index.core.agent.workflow to create an agent with a query engine tool.

In [ ]:
# 上記のエラーとなったため、再構築
import os
from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai import OpenAI

from llama_index.core.tools import QueryEngineTool
from llama_index.core.agent.workflow import FunctionAgent

# 環境変数の取得
load_dotenv("../.env")
os.environ["OPENAI_API_KEY"] = os.environ["API_KEY"]  # 既存コード踏襲（どちらか片方でもOK）

# モデル名
MODEL_NAME = "gpt-4o-mini"

# ドキュメントからテキスト情報を読込
documents = SimpleDirectoryReader("./data/text").load_data()

# インデックスの構築
index = VectorStoreIndex.from_documents(documents)

# ここが旧 as_chat_engine の置き換え ----------------------------

# 1) LLM
llm = OpenAI(model=MODEL_NAME)

# 2) index -> query engine
query_engine = index.as_query_engine(similarity_top_k=5)

# 3) query engine をツール化
tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="doc_search",
    description="指定フォルダの文書を検索して回答します。",
)

# 4) Agent（= 新しい “チャットエンジン” 的存在）を作成
chat_engine = FunctionAgent(
    llm=llm,
    tools=[tool],
    verbose=True,
)

# 使い方例（Notebookは await が基本）
response = await chat_engine.run("このフォルダの文書の概要を教えて")
print(str(response))

2026-02-24 13:53:11,155 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-24 13:53:12,336 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-24 13:53:12,906 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-24 13:53:15,593 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-24 13:53:16,226 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


フォルダ内の文書は、会社の規則や方針に関する内容が含まれています。具体的には以下のようなトピックが扱われています：

- 会社資産の使用
- セキュリティ対策
- 出張費の支給
- 出張中の勤務時間と安全確保
- 社内規則の改定・見直し
- 従業員の意見表明と改善提案
- 知的財産に関する規則
- 行動規範・倫理規程
- 情報の管理と守秘義務
- 不正行為の防止
- 公私の区別
- 健康・安全管理規則
- その他の特定事項（リモートワーク制度やフレックスタイム制度など）

これらの文書は、会社の運営や従業員の行動に関する重要なガイドラインを提供しています。


In [17]:
# 質問：1回目
response = query_engine.query("有給休暇はいつから取得できますか？")

# 言語モデルからの回答を表示
print(str(response))

2026-02-24 13:39:52,407 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-24 13:39:53,624 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


有給休暇は、入社から6ヶ月継続勤務し、全労働日の8割以上出勤した場合に初めて取得できます。


In [18]:
# 引用元を表示
for sn in response.source_nodes:
    print("ファイル名：", sn.node.metadata.get("file_name"))
    print("関連度スコア:", sn.score)
    print("テキスト：")
    print(sn.node.text)
    print("-" * 50)

ファイル名： 03休暇規則.md
関連度スコア: 0.8744311031347539
テキスト：
**産前産後休暇**

   - **産前休暇**：出産予定日の**6週間前**から取得可能です。
   - **産後休暇**：出産日の翌日から**8週間**は就業が禁止されています。
   - 産前産後休暇中は、健康保険から出産手当金が支給されます。

3. **育児休業**

   - 子供が**1歳**になるまでの間、育児休業を取得できます。
   - 保育所に入れないなどの事情がある場合、最長で**2歳**まで延長可能です。
   - 育児休業中は、雇用保険から育児休業給付金が支給されます。

4. **介護休業**

   - 要介護状態にある家族を介護するために、**通算93日間**の介護休業を取得できます。
   - 介護休業は、対象家族一人につき1回、分割して最大3回まで取得可能です。

5. **生理休暇**

   - 女性従業員で、生理により就業が困難な場合は、申請により休暇を取得できます。

6. **裁判員休暇**

   - 裁判員や補充裁判員として選任された場合、その期間中は休暇を取得できます。

### 3. 特別有給休暇

会社が特別に認めた有給の休暇です。

1. **リフレッシュ休暇**

   - **勤続5年**ごとに、連続した**5日間**のリフレッシュ休暇が取得できます。
   - リフレッシュ休暇は、有給休暇とは別に付与されます。

2. **ボランティア休暇**

   - 社会貢献活動を支援するため、年間**2日間**のボランティア休暇を取得できます。
   - ボランティア休暇を取得する際は、活動内容を事前に上司へ報告してください。

### 4. 無給休暇

給与の支給がない休暇です。

1. **自己啓発休業**

   - 自己啓発や留学などの目的で、最長**2年間**の休業が可能です。
   - 休業期間中は、社会保険料の自己負担などが発生します。

2. **私傷病休業**

   - 病気やけがで長期間の治療が必要な場合、最長**1年間**の休業が可能です。
   - 休業中は、健康保険から傷病手当金が支給される場合があります。

### 5. 休暇取得の手続き

1.
----------------

In [22]:
# 質問：2回目
response = query_engine.query("勤続年数が5年の場合は何日ですか？")

# 言語モデルからの回答を表示
print(str(response))

2026-02-24 13:53:32,574 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-24 13:53:33,782 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


11日


In [23]:
response = await chat_engine.run("勤続年数が5年の場合は何日ですか？")
print(str(response))

2026-02-24 13:54:05,375 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-24 13:54:05,875 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-24 13:54:06,827 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-24 13:54:07,369 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


勤続年数が5年の場合は18日です。


In [24]:
# 質問：1回目(再)
response = query_engine.query("有給休暇はいつから取得できますか？")

# 言語モデルからの回答を表示
print(str(response))

2026-02-24 13:59:16,923 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-24 13:59:18,146 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


有給休暇は入社から6ヶ月継続勤務し、全労働日の8割以上出勤した場合に初めて取得できます。


In [25]:
response = await chat_engine.run("有給休暇はいつから取得できますか？")
print(str(response))

2026-02-24 13:59:40,991 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-24 13:59:41,420 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-24 13:59:42,662 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-24 13:59:43,468 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


有給休暇は、原則として3日前までに上司に申請することで取得できます。


In [28]:
response = query_engine.query("勤続年数が5年の場合は何日ですか？")
print(str(response))

2026-02-24 14:05:29,253 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-24 14:05:30,256 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


勤続年数が5年の場合、年次有給休暇日数は18日です。


In [29]:
for sn in response.source_nodes:
    print("ファイル名：", sn.node.metadata.get("file_name"))
    print("関連度スコア:", sn.score)
    print("テキスト：")
    print(sn.node.text)
    print("-" * 50)

ファイル名： 03休暇規則.md
関連度スコア: 0.7928090398993632
テキスト：
**産前産後休暇**

   - **産前休暇**：出産予定日の**6週間前**から取得可能です。
   - **産後休暇**：出産日の翌日から**8週間**は就業が禁止されています。
   - 産前産後休暇中は、健康保険から出産手当金が支給されます。

3. **育児休業**

   - 子供が**1歳**になるまでの間、育児休業を取得できます。
   - 保育所に入れないなどの事情がある場合、最長で**2歳**まで延長可能です。
   - 育児休業中は、雇用保険から育児休業給付金が支給されます。

4. **介護休業**

   - 要介護状態にある家族を介護するために、**通算93日間**の介護休業を取得できます。
   - 介護休業は、対象家族一人につき1回、分割して最大3回まで取得可能です。

5. **生理休暇**

   - 女性従業員で、生理により就業が困難な場合は、申請により休暇を取得できます。

6. **裁判員休暇**

   - 裁判員や補充裁判員として選任された場合、その期間中は休暇を取得できます。

### 3. 特別有給休暇

会社が特別に認めた有給の休暇です。

1. **リフレッシュ休暇**

   - **勤続5年**ごとに、連続した**5日間**のリフレッシュ休暇が取得できます。
   - リフレッシュ休暇は、有給休暇とは別に付与されます。

2. **ボランティア休暇**

   - 社会貢献活動を支援するため、年間**2日間**のボランティア休暇を取得できます。
   - ボランティア休暇を取得する際は、活動内容を事前に上司へ報告してください。

### 4. 無給休暇

給与の支給がない休暇です。

1. **自己啓発休業**

   - 自己啓発や留学などの目的で、最長**2年間**の休業が可能です。
   - 休業期間中は、社会保険料の自己負担などが発生します。

2. **私傷病休業**

   - 病気やけがで長期間の治療が必要な場合、最長**1年間**の休業が可能です。
   - 休業中は、健康保険から傷病手当金が支給される場合があります。

### 5. 休暇取得の手続き

1.
----------------

In [ ]:
# llama-index のバージョン確認
import importlib.metadata as md

pkgs = [
    "llama-index",
    "llama-index-core",
    "llama-index-llms-openai",
]
for p in pkgs:
    try:
        print(p, md.version(p))
    except Exception:
        print(p, "not installed")

llama-index 0.14.15
llama-index-core 0.14.15
llama-index-llms-openai 0.6.19
